[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badaouihakimou/machine-learning-notebooks/blob/main/06_list_comprehension.ipynb)



# Les compréhensions

Construire une liste avec une boucle demande trois lignes : créer la liste vide,
boucler, ajouter. La compréhension fait la même chose en une seule.

Ce n'est pas qu'une question de concision c'est aussi plus rapide, et c'est la
syntaxe qu'on retrouve partout en Python moderne.

## Le plan

| Section | Le sujet |
|---|---|
| 1 | La compréhension de liste |
| 2 | Pourquoi c'est plus rapide |
| 3 | Filtrer avec `if` |
| 4 | Les compréhensions imbriquées |
| 5 | Compréhension de dictionnaire |
| 6 | Les parenthèses ne font pas un tuple |
| 7 | Quand ne pas l'utiliser |

## Deux pièges annoncés

Le premier concerne les accolades : `{'a', 'b'}` n'est pas un dictionnaire mais
un ensemble, et un ensemble ne conserve pas l'ordre. Pire, cet ordre change
d'une exécution à l'autre.

Le second concerne les parenthèses : `(i for i in range(10))` ne produit pas un
tuple mais un générateur, qui se vide après usage.

Prérequis : les notebooks 03 à 05.

## 1. La compréhension de liste

Voici le schéma qu'on écrit tout le temps : une liste vide, une boucle, un
`append`.

In [1]:
liste_1 = []
for i in range(10):
    liste_1.append(i**2)

print(liste_1)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


La compréhension fait exactement la même chose en une ligne.

In [2]:
liste_2 = [i**2 for i in range(10)]
print(liste_2)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]


### Comment la lire

```
[  i**2        for i in range(10)  ]
   ↑           ↑
   ce qu'on    d'où viennent
   produit     les valeurs
```

L'ordre d'écriture n'est pas l'ordre d'exécution. Le `for` s'exécute d'abord,
l'expression de gauche s'évalue à chaque tour.

Pour la construire, pars toujours de la boucle et déplace le contenu de
l'`append` vers la gauche :

```python
for i in range(10):          →    [    for i in range(10)]
    liste.append(i**2)       →    [i**2 for i in range(10)]
```

### Sur n'importe quel itérable

Comme la boucle `for`, la compréhension accepte tout ce qui se parcourt.

In [3]:
villes = ['paris', 'tokyo', 'nice']

print([v.upper() for v in villes])
print([len(v) for v in villes])
print([lettre for lettre in 'Python'])
print([n * 2 for n in (1, 2, 3)])

['PARIS', 'TOKYO', 'NICE']
[5, 5, 4]
['P', 'y', 't', 'h', 'o', 'n']
[2, 4, 6]


## 2. Pourquoi c'est plus rapide

Mesurons sur dix millions d'éléments.

In [5]:
import time

debut = time.time()
liste_1 = []
for i in range(1_000_000):
    liste_1.append(i**2)
boucle = time.time() - debut

debut = time.time()
liste_2 = [i**2 for i in range(1_000_000)]
comprehension = time.time() - debut

print(f'Boucle : {boucle:.3f} s')
print(f'Compréhension : {comprehension:.3f} s')
print(f'Gain : {boucle/comprehension:.1f}x')

Boucle : 0.333 s
Compréhension : 0.198 s
Gain : 1.7x


L'écart est d'environ 30 à 50 %. Modeste mais réel.

D'où vient le gain. Dans la boucle, Python doit à chaque tour retrouver
l'attribut `append` de la liste, puis appeler cette méthode. Un million de fois.
La compréhension utilise une instruction interne dédiée qui évite ce va-et-vient.

Ce n'est donc pas magique, et ça reste du Python : pour un vrai gain de vitesse
sur des nombres, c'est NumPy qu'il faut, avec un facteur de l'ordre de 50.

```python
import numpy as np
resultat = np.arange(1_000_000) ** 2
```

La vraie raison d'utiliser une compréhension n'est pas la vitesse, c'est la
lisibilité. Une ligne qui dit « la liste des carrés de 0 à 9 » se lit d'un coup,
alors que trois lignes obligent à reconstruire l'intention.

## 3. Filtrer avec if

Un `if` en fin de compréhension ne garde que les éléments qui satisfont la
condition.

In [6]:
pairs = [n for n in range(20) if n % 2 == 0]
print(pairs)

longues = [v for v in ['paris', 'tokyo', 'nice', 'rio'] if len(v) > 4]
print(longues)

[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
['paris', 'tokyo']


L'équivalent en boucle :

```python
pairs = []
for n in range(20):
    if n % 2 == 0:
        pairs.append(n)
```

Même ordre de lecture : le `for`, puis le `if`, puis ce qu'on produit.

### Le if ternaire : à ne pas confondre

Il existe une autre position pour un `if`, et elle ne fait pas du tout la même
chose.

In [7]:
nombres = range(10)

filtre = [n for n in nombres if n % 2 == 0]
print('if à la fin   :', filtre, '-> filtre, 5 éléments')

transforme = ['pair' if n % 2 == 0 else 'impair' for n in nombres]
print('if au début   :', transforme, '-> transforme, 10 éléments')

if à la fin   : [0, 2, 4, 6, 8] -> filtre, 5 éléments
if au début   : ['pair', 'impair', 'pair', 'impair', 'pair', 'impair', 'pair', 'impair', 'pair', 'impair'] -> transforme, 10 éléments


| Position | Rôle | Longueur du résultat |
|---|---|---|
| `if` à la fin | filtre | réduite |
| `if ... else` au début | transforme | inchangée |

Le second est un opérateur ternaire : `valeur_si_vrai if condition else valeur_si_faux`. Le `else` y est obligatoire, contrairement au filtre.

Les deux se combinent :

In [8]:
resultat = ['grand' if n > 5 else 'petit' for n in range(10) if n % 2 == 0]
print(resultat)

['petit', 'petit', 'petit', 'grand', 'grand']


Ça marche, mais on approche de la limite de lisibilité. Au-delà, une boucle
classique est préférable.

## 4. Les compréhensions imbriquées

Une compréhension peut en contenir une autre, ce qui produit une liste de listes.

In [9]:
grille = [[i for i in range(3)] for j in range(3)]
print(grille)

[[0, 1, 2], [0, 1, 2], [0, 1, 2]]


La compréhension intérieure `[i for i in range(3)]` est évaluée trois fois, une
par valeur de `j`. Comme elle ne dépend pas de `j`, les trois lignes sont
identiques.

En faisant intervenir les deux variables, chaque ligne devient différente :

In [10]:
grille = [[i + j for i in range(3)] for j in range(3)]

for ligne in grille:
    print(ligne)

[0, 1, 2]
[1, 2, 3]
[2, 3, 4]


C'est une table d'addition. La table de multiplication s'écrit pareil :

In [11]:
table = [[i * j for i in range(1, 6)] for j in range(1, 6)]

for ligne in table:
    print(' '.join(f'{v:3}' for v in ligne))

  1   2   3   4   5
  2   4   6   8  10
  3   6   9  12  15
  4   8  12  16  20
  5  10  15  20  25


### Un point important sur les listes imbriquées

Chaque ligne est une liste indépendante, contrairement à ce que produirait
une multiplication.

In [12]:
avec_comprehension = [[0] * 3 for _ in range(3)]
avec_multiplication = [[0] * 3] * 3 # DANGER

avec_comprehension[0][0] = 99
avec_multiplication[0][0] = 99

print('compréhension  :', avec_comprehension)
print('multiplication :', avec_multiplication)

compréhension  : [[99, 0, 0], [0, 0, 0], [0, 0, 0]]
multiplication : [[99, 0, 0], [99, 0, 0], [99, 0, 0]]


La seconde version modifie les trois lignes. C'est encore le partage d'objets
mutables des notebooks 02, 04 et 05 : `[liste] * 3` répète la même liste
trois fois.

La compréhension, elle, évalue `[0] * 3` à chaque tour et crée une liste neuve.

C'est le troisième notebook où ce piège apparaît. À force, la règle devient
claire : dès qu'on veut plusieurs objets mutables, il faut les créer un par un,
jamais les dupliquer.

### Aplatir une liste de listes

Deux `for` à la suite parcourent en profondeur au lieu d'imbriquer.

In [13]:
imbriquee = [[1, 2, 3], [4, 5], [6]]

plate = [element for sous_liste in imbriquee for element in sous_liste]
print(plate)

[1, 2, 3, 4, 5, 6]


L'ordre des deux `for` est celui de la boucle imbriquée équivalente :

```python
plate = []
for sous_liste in imbriquee:
    for element in sous_liste:
        plate.append(element)
```

De gauche à droite, comme on l'écrirait de haut en bas. C'est contre-intuitif la
première fois, parce qu'on s'attend à lire de l'intérieur vers l'extérieur.

## 5. Compréhension de dictionnaire

Même principe, avec des accolades et un `:` entre la clé et la valeur.

In [14]:
carres = {n: n**2 for n in range(1, 6)}
print(carres)

{1: 1, 2: 4, 3: 9, 4: 16, 5: 25}


### Le piège des accolades

Attention : les accolades servent aussi aux ensembles. Sans les deux-points,
ce n'est plus un dictionnaire.

In [15]:
un_dictionnaire = {'Pierre': 1, 'Jean': 2}
un_ensemble = {'Pierre', 'Jean', 'Julie', 'Sophie'}

print(type(un_dictionnaire))
print(type(un_ensemble))

<class 'dict'>
<class 'set'>


Un ensemble ressemble à une liste sans doublons, mais avec une propriété
essentielle : il n'a pas d'ordre.

Et ce n'est pas seulement que l'ordre diffère de celui d'écriture il change
d'une exécution du programme à l'autre.

In [16]:
prenoms_set = {'Pierre', 'Jean', 'Julie', 'Sophie'}

print('Ordre observé :', list(prenoms_set))
print()
print("Relance le NOYAU (pas juste la cellule) et réexécute :")
print("l'ordre sera probablement différent.")

Ordre observé : ['Pierre', 'Jean', 'Julie', 'Sophie']

Relance le NOYAU (pas juste la cellule) et réexécute :
l'ordre sera probablement différent.


Python randomise le hachage des chaînes à chaque démarrage, pour des raisons de
sécurité. L'ordre d'un ensemble de chaînes est donc imprévisible d'une session à
l'autre.

La conséquence est sérieuse pour un notebook publié. Si tu construis un
dictionnaire à partir d'un ensemble :

```python
dico = {k: v for k, v in enumerate(prenoms_set)}
```

Les numéros attribués changeront à chaque redémarrage du noyau. Le notebook
donnera un résultat différent de celui affiché, sans qu'aucune erreur ne
survienne.

**La correction** tient à un caractère : des crochets au lieu des accolades.

In [18]:
prenoms = ['Pierre', 'Jean', 'Julie', 'Sophie'] # une liste

dico = {k: v for k, v in enumerate(prenoms)}
print(dico)

{0: 'Pierre', 1: 'Jean', 2: 'Julie', 3: 'Sophie'}


Maintenant l'ordre est garanti, à chaque exécution.

Quand utiliser un ensemble. Il est très utile pour tester l'appartenance et
supprimer les doublons, opérations où il est bien plus rapide qu'une liste. Mais
jamais quand l'ordre compte.

### Construire un dictionnaire depuis deux listes

In [19]:
prenoms = ['Pierre', 'Jean', 'Julie', 'Sophie']
ages = [10, 20, 30, 40]

dico = {p: a for p, a in zip(prenoms, ages)}
print(dico)

{'Pierre': 10, 'Jean': 20, 'Julie': 30, 'Sophie': 40}


Rappel du notebook 04 : `zip` s'arrête à la plus courte des deux listes, sans
message. Vérifie les longueurs si elles viennent de sources différentes.

Pour ce cas précis, `dict(zip(...))` fait la même chose sans compréhension :

```python
dico = dict(zip(prenoms, ages))
```

### Filtrer un dictionnaire

In [20]:
majeurs = {p: a for p, a in zip(prenoms, ages) if a > 18}
print(majeurs)

{'Jean': 20, 'Julie': 30, 'Sophie': 40}


Attention à l'ordre des variables. Ici `p` reçoit un prénom et `a` un âge,
parce que `zip(prenoms, ages)` est dans cet ordre. Inverser le `zip` sans
inverser le filtre produit un `TypeError` :

```python
{k: v for k, v in zip(ages, prenoms) if v > 18}
# TypeError : '>' not supported between 'str' and 'int'
```

Des noms de variables parlants `p` et `a` plutôt que `k` et `v` évitent
cette confusion.

## 6. Les parenthèses ne font pas un tuple

Il n'existe pas de compréhension de tuple. Les parenthèses produisent tout
autre chose.

In [21]:
resultat = (i for i in range(10))
print(resultat)
print(type(resultat))

<generator object <genexpr> at 0x7da8b65ba8c0>
<class 'generator'>


C'est un générateur. Il ne contient aucune valeur : il sait comment les
calculer, une par une, à la demande.

Pour obtenir un vrai tuple, il faut convertir explicitement :

In [22]:
mon_tuple = tuple(i for i in range(10))
print(mon_tuple, type(mon_tuple))

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9) <class 'tuple'>


### À quoi sert un générateur

À économiser la mémoire. Il ne stocke rien.

In [23]:
import sys

liste = [i**2 for i in range(100_000)]
generateur = (i**2 for i in range(100_000))

print('Liste:', sys.getsizeof(liste), 'octets')
print('Générateur :', sys.getsizeof(generateur), 'octets')

Liste: 800984 octets
Générateur : 200 octets


Quelques centaines de milliers d'octets contre moins de deux cents.

C'est le même principe que `range`, vu au notebook 03 : `range(10_000_000)`
occupe 48 octets parce qu'il calcule au lieu de stocker.

Utile quand on veut juste parcourir sans conserver :

```python
total = sum(i**2 for i in range(1_000_000)) # pas de liste intermédiaire
```

### Le piège : un générateur s'épuise

Une fois parcouru, il est vide. Définitivement.

In [24]:
generateur = (i for i in range(5))

print('Premier parcours :', sum(generateur))
print('Second parcours  :', sum(generateur))

Premier parcours : 10
Second parcours  : 0


Zéro au second appel. Aucune erreur, juste un résultat faux.

C'est le comportement le plus déroutant des générateurs. Si tu dois parcourir
plusieurs fois, prends une liste.

**En résumé** :

| Écriture | Produit |
|---|---|
| `[x for x in ...]` | une liste |
| `{x: y for ...}` | un dictionnaire |
| `{x for x in ...}` | un ensemble |
| `(x for x in ...)` | un générateur |
| `tuple(x for x in ...)` | un tuple |

## 7. Quand ne pas utiliser une compréhension

### Quand elle devient illisible

Le but est la clarté. Si la ligne ne se lit plus d'un coup, la boucle est
meilleure.

```python
# Illisible
resultat = [f(x) if cond1(x) else g(x) for sous in data for x in sous if cond2(x) and cond3(x)]
```

Une règle simple : si tu ne peux pas expliquer la ligne en une phrase, découpe-la.

### Quand il n'y a pas de résultat à produire

Une compréhension construit quelque chose. Si tu veux seulement provoquer un
effet, utilise une boucle.

```python
# Mauvais : construit une liste de None qu'on jette
[print(x) for x in liste]

# Correct
for x in liste:
    print(x)
```

Le premier fonctionne, mais il crée en mémoire une liste inutile un problème
réel sur de gros volumes.

### Quand il faut construire plusieurs choses

```python
# Deux compréhensions parcourent deux fois
pairs = [n for n in nombres if n % 2 == 0]
impairs = [n for n in nombres if n % 2 != 0]

# Une seule boucle suffit
pairs, impairs = [], []
for n in nombres:
    (pairs if n % 2 == 0 else impairs).append(n)
```

### Quand NumPy fait mieux

Sur des calculs numériques, la compréhension reste du Python.

In [25]:
import numpy as np
import time

n = 1_000_000

debut = time.time()
resultat_py = [i**2 for i in range(n)]
temps_py = time.time() - debut

debut = time.time()
resultat_np = np.arange(n) ** 2
temps_np = time.time() - debut

print(f'Compréhension : {temps_py:.4f} s')
print(f'NumPy : {temps_np:.4f} s')
print(f'Rapport : {temps_py/temps_np:.0f}x')

Compréhension : 0.0914 s
NumPy : 0.0088 s
Rapport : 10x


Un facteur de plusieurs dizaines. Sur des nombres, NumPy gagne toujours.

La compréhension reste le bon outil pour du texte, des objets, des structures
mixtes tout ce que NumPy ne sait pas vectoriser.

## 8. Une note sur les noms de variables

Un commentaire fréquent affirme que `list`, `dict` et `tuple` sont des « termes
protégés » qu'on ne peut pas utiliser comme noms de variables.

C'est inexact, et la nuance a des conséquences pratiques.

In [26]:
list = [1, 2, 3]  # accepté, aucune erreur
print('list vaut :', list)

try:
    autre = list(range(5))
except TypeError as e:
    print('TypeError :', e)

list vaut : [1, 2, 3]
TypeError : 'list' object is not callable


Python accepte l'affectation. Mais la fonction `list` est perdue pour le reste
de la session, et tout appel ultérieur échoue avec un message qui n'aide pas
beaucoup.

Nettoyons avant de continuer :

In [27]:
del list # restaure la fonction d'origine
print(list(range(5)))

[0, 1, 2, 3, 4]


La distinction. Les vrais mots-clés protégés `if`, `for`, `def`, `class`,
`return`, `lambda` lèvent une `SyntaxError` immédiate. `list`, `dict`, `str`,
`sum`, `min`, `max`, `type`, `id`, `input` sont des fonctions intégrées, qu'on
peut écraser sans avertissement.

Pour voir la liste complète :

```python
print(dir(__builtins__))
```

En pratique, ajoute un underscore ou choisis un nom parlant : `ma_liste`,
`liste_1`, `noms`. Les éditeurs modernes colorent différemment les fonctions
intégrées, ce qui aide à repérer l'écrasement.

C'est un bug particulièrement pénible parce que l'erreur apparaît loin de sa
cause parfois dans une cellule qui marchait très bien avant.

## 9. Mémo

### Les quatre écritures

```python
[expression for x in iterable] # liste
[expression for x in iterable if condition] # liste filtrée
{cle: valeur for x in iterable} # dictionnaire
{expression for x in iterable} # ensemble
(expression for x in iterable) # générateur, pas un tuple
```

### if avant ou après

| Position | Rôle |
|---|---|
| `[x for x in l if cond]` | filtre, le `else` est interdit |
| `[a if cond else b for x in l]` | transforme, le `else` est obligatoire |

### Les pièges

| Situation | Ce qui se passe |
|---|---|
| `{'a', 'b'}` | un ensemble, pas un dictionnaire |
| Ordre d'un ensemble | change à chaque redémarrage du noyau |
| `(x for x in ...)` | un générateur, pas un tuple |
| Générateur parcouru deux fois | vide au second passage |
| `[[0]*3]*3` | les trois lignes sont le même objet |
| `[print(x) for x in l]` | construit une liste de `None` inutile |
| `list = [...]` | écrase la fonction `list` |
| `zip` sur longueurs inégales | des éléments disparaissent |

## 10. Exercices

**Exercice 1**

Écris en une compréhension : la liste des nombres de 1 à 100 divisibles par 3
mais pas par 5. Puis la même chose en boucle classique, et compare les temps
d'exécution sur 1 à 1 000 000.

**Exercice 2**

À partir d'une liste de prénoms, construis un dictionnaire qui associe chaque
prénom à sa longueur, uniquement pour les prénoms de plus de 4 lettres. Puis
inverse-le pour associer chaque longueur à la liste des prénoms concernés.

Attention, la seconde partie ne se fait pas en une compréhension simple —
demande-toi pourquoi.

**Exercice 3**

Cette ligne contient le piège du partage d'objets :

```python
matrice = [[0] * 3] * 3
```

Montre le problème, explique-le, corrige-le avec une compréhension. Puis écris
une compréhension qui produit la matrice identité 4×4, celle qui a des 1 sur la
diagonale et des 0 ailleurs.

## Pour continuer

Le notebook suivant porte sur les fonctions intégrées de Python : `map`,
`filter`, `sorted`, `zip`, `enumerate` et les autres. Plusieurs font double
emploi avec les compréhensions, et on verra laquelle choisir.

Les compréhensions reviendront constamment. En Pandas, `[col for col in
df.columns if df[col].dtype == 'object']` est l'idiome pour repérer les colonnes
textuelles.

In [28]:
# Exerice 1

In [29]:
resultat = [n for n in range(1, 101) if n % 3 == 0 and n % 5 != 0]

print(resultat)
print(len(resultat), 'nombres')

[3, 6, 9, 12, 18, 21, 24, 27, 33, 36, 39, 42, 48, 51, 54, 57, 63, 66, 69, 72, 78, 81, 84, 87, 93, 96, 99]
27 nombres


In [31]:
resultat = []
for n in range(1, 101):
    if n % 3 == 0 and n % 5 != 0:
        resultat.append(n)
resultat

[3,
 6,
 9,
 12,
 18,
 21,
 24,
 27,
 33,
 36,
 39,
 42,
 48,
 51,
 54,
 57,
 63,
 66,
 69,
 72,
 78,
 81,
 84,
 87,
 93,
 96,
 99]

In [32]:
import time

def avec_boucle(n):
    res = []
    for i in range(1, n + 1):
        if i % 3 == 0 and i % 5 != 0:
            res.append(i)
    return res

def avec_comprehension(n):
    return [i for i in range(1, n + 1) if i % 3 == 0 and i % 5 != 0]

N = 1_000_000

debut = time.time(); avec_boucle(N);        t_boucle = time.time() - debut
debut = time.time(); avec_comprehension(N); t_comp = time.time() - debut

print(f'Boucle : {t_boucle:.3f} s')
print(f'Compréhension : {t_comp:.3f} s')
print(f'Gain : {t_boucle/t_comp:.1f}x')

Boucle : 0.077 s
Compréhension : 0.136 s
Gain : 0.6x


In [33]:
resultat = [n for n in range(3, 101, 3) if n % 5 != 0]
resultat

[3,
 6,
 9,
 12,
 18,
 21,
 24,
 27,
 33,
 36,
 39,
 42,
 48,
 51,
 54,
 57,
 63,
 66,
 69,
 72,
 78,
 81,
 84,
 87,
 93,
 96,
 99]

In [34]:
# Exerice 2

prenoms = ['Ali', 'Sara', 'Yanis', 'Nour', 'Lina', 'Omar', 'Ines', 'Meriem']

longueurs = {p: len(p) for p in prenoms if len(p) > 4}
print(longueurs)

{'Yanis': 5, 'Meriem': 6}


In [35]:
inverse = {longueur: prenom for prenom, longueur in longueurs.items()}
print(inverse)

{5: 'Yanis', 6: 'Meriem'}


In [36]:
def inverser_par_longueur(dictionnaire):
    """Associe chaque longueur à la liste des prénoms concernés."""
    resultat = {}
    for prenom, longueur in dictionnaire.items():
        resultat.setdefault(longueur, []).append(prenom)
    return resultat

print(inverser_par_longueur(longueurs))

{5: ['Yanis'], 6: ['Meriem']}


In [37]:
inverse = {l: [p for p in longueurs if len(p) == l] for l in set(longueurs.values())}

In [38]:
# Exerice 3

In [39]:
matrice = [[0] * 3] * 3
print(matrice)

matrice[0][0] = 99
print(matrice)

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]
[[99, 0, 0], [99, 0, 0], [99, 0, 0]]


In [40]:
matrice = [[0] * 3] * 3
print('ligne 0 et ligne 1, même objet ?', matrice[0] is matrice[1])
print('id ligne 0 :', id(matrice[0]))
print('id ligne 1 :', id(matrice[1]))

ligne 0 et ligne 1, même objet ? True
id ligne 0 : 138163240823168
id ligne 1 : 138163240823168


In [41]:
matrice = [[0] * 3 for _ in range(3)]

matrice[0][0] = 99
print(matrice)
print('même objet ?', matrice[0] is matrice[1])

[[99, 0, 0], [0, 0, 0], [0, 0, 0]]
même objet ? False


In [42]:
identite = [[1 if i == j else 0 for i in range(4)] for j in range(4)]

for ligne in identite:
    print(ligne)

[1, 0, 0, 0]
[0, 1, 0, 0]
[0, 0, 1, 0]
[0, 0, 0, 1]


In [43]:
import numpy as np
print(np.eye(4))
print(np.zeros((3, 3)))

[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
